In [1]:
# import modules

import os
import pandas as pd
from docx import Document
import re

import pandas as pd

import os
import pandas as pd
# from pyth.plugins.plaintext.writer import PlaintextWriter


from striprtf.striprtf import rtf_to_text

from datetime import datetime
import locale



In [2]:
# Set the locale to German
locale.setlocale(locale.LC_TIME, 'de_DE')

'de_DE'

# 1 Define all the necessary functions

In [3]:
# This function reads all the nmaenames of the folders in a parent folder(i.e the country names)
def get_country_names(parent_folder):
    country_names = []
    
    for item in os.listdir(parent_folder):
        item_path = os.path.join(parent_folder, item)
        if os.path.isdir(item_path):
            country_names.append(item)

    return country_names

In [4]:
# For each country in the parent folder, this fucntion combines all the individual nws files as a list
def combine_files_for_country(parent_folder, country):
    # i = 0
    all_files = []
    country_folder = os.path.join(parent_folder, country)
    
    # Check if the directory exists
    if not os.path.exists(country_folder):
        print(f"Error: Directory '{country_folder}' does not exist.")
        return ""

    # Iterate through all RTF files in the country folder
    for root, dirs, files in os.walk(country_folder):
        print(f"processing {len(files)} files for {country}")
        for file in files:
            if file.endswith(".rtf"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as file:
                    document_text = file.read()

                    all_files.append(document_text)

    return all_files


In [5]:
# for each file, this function converts the rtf to plain text and extract the article texts
def get_articles(articles):
    plain_text = rtf_to_text(articles)

    # Updated regular expression pattern
    pattern = r'(.*?)(Dokument [A-Za-z0-9]{8}\d{8}[A-Za-z0-9]{9}|$)'

    # Use re.findall with the pattern
    matches = re.findall(pattern, plain_text, re.DOTALL)

    # Extract content from matches
    article_texts = [match[0].strip() for match in matches if match[0].strip()]

    return article_texts

In [6]:
# for each article in a file, this function extracts the article title.
# The title is usually the last line in the first "paragragh"

def extract_title(paragraph):
    lines = paragraph.splitlines()
    if lines:
        title =lines[-1]

    else:
            title = "unknown"
    return title   

In [7]:
# for each article in a file, this function extracts the article date.
# The date is usually somewhere in the second "paragragh"
def extract_date(paragraph):
    # Define a regular expression pattern for the date
    date_pattern = re.compile(r'\b(\d{1,2}\s+(Januar|Februar|März|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember)\s+\d{4})\b', flags=re.IGNORECASE | re.UNICODE)


    # Search for the date pattern in the paragraph
    match = date_pattern.search(paragraph)

    # If a match is found, extract the matched date string
    if match:
        date_str = match.group(1)

        # Convert the date string to a datetime object
        try:
            date = datetime.strptime(date_str, "%d %B %Y")
       
        except ValueError:
            date =  None

    return date

In [8]:
# after extracting all the article info for a country, we use this function to create a dataframe and save it to disk in csv format
def save_file(result_dict, country, output_folder_path):
    df= pd.DataFrame(result_dict)
    df= df[df["title"] != "Zusammenfassung der Suche"].reset_index(drop=True)

    # for country names that consist of multiple words, we replace the empty space with an underscore
    country = country.replace(" ", "_")
    
    # save to the 'output' folder

    #specify the output path
    output_path = output_folder_path + "/" + country + ".csv" 
    df.to_csv(output_path, index=False)

# 2. Read the input files and call the functions above one after the other

In [9]:
# specify the parent folder
parent_folder = "./countries"
countries = get_country_names(parent_folder)

output_folder_path = "./output"


for index, country in enumerate(countries):  

    result = []
    all_files = combine_files_for_country(parent_folder, country)

    for file in all_files:

    # combined_article = combine_articles_for_country(parent_folder, country)
        article_texts = get_articles(file)
        # print(f"Total articles: {len(article_texts)}")

        for article in article_texts:

            # Split the article into paragraphs
            paragraphs = article.split('\n\n')

            # Print the first paragraph
            if paragraphs:
                try:
                    first_paragraph = paragraphs[0]
                    title = extract_title(first_paragraph)
                except:
                    title = None

                try:
                    second_paragraph = paragraphs[1]
                    date = extract_date(second_paragraph)
                except:
                    date = None

                try:
                    text = '\n\n'.join(paragraphs[2:])
                except:
                    text = None

            else:
                title= None
                date = None
                text = None
            
            result.append({'title':title,'date':date,'text':text})


    save_file(result, country,output_folder_path)
    print(f"country {index+1} of {len(countries)} completed" )
    print("-----------------------------")

processing 7 files for Argentina


country 1 of 48 completed
-----------------------------
processing 4 files for Australia
country 2 of 48 completed
-----------------------------
processing 21 files for Brazil
country 3 of 48 completed
-----------------------------
processing 1 files for Bulgaria
country 4 of 48 completed
-----------------------------
processing 7 files for Canada
country 5 of 48 completed
-----------------------------
processing 8 files for Chile
country 6 of 48 completed
-----------------------------
processing 29 files for China
country 7 of 48 completed
-----------------------------
processing 4 files for Colombia
country 8 of 48 completed
-----------------------------
processing 1 files for Croatia
country 9 of 48 completed
-----------------------------
processing 1 files for Czech Republic
country 10 of 48 completed
-----------------------------
processing 2 files for Denmark
country 11 of 48 completed
-----------------------------
processing 12 files for Hongkong
country 12 of 48 completed
-----

In [16]:
united_states

,title,date,text
0,JPMorgan Chase & Co. - Primary Offering Prospe...,2023-08-21,Access the original document here\n\nPrimary O...
1,Bunge Ltd. - Quarterly Report for Quarter Endi...,2023-08-09,Access the original document here\n\nQuarterly...
2,Capital Group Core Plus Income ETF - Post-Effe...,2023-08-08,Access the original document here\n\nPost-Effe...
3,AB Active ETFs Inc. - Post-Effective Amendment...,2023-08-07,Access the original document here\n\nPost-Effe...
4,Kiniksa Pharmaceuticals Ltd. - Quarterly Repor...,2023-08-01,Access the original document here\n\nQuarterly...
...,...,...,...
852,Dollar ends lower as German mark strengthens.,1995-09-28,NEW YORK (Reuter) - The dollar edged lower ag...
853,Dollar tumbles as rally on German rate cut fiz...,1995-08-24,NEW YORK (Reuter) - The dollar tumbled from a...
854,"Dollar ends mixed, waits for Fed, Bundesbank m...",1995-08-21,NEW YORK (Reuter) - The dollar ended mixed Mo...
855,"Japan reproaches U.S. over dollar, urges action.",1995-04-20,"TOKYO, April 20 (Reuter) - Japan reproached t..."


In [13]:
united_states = pd.read_csv("./output/United_States.csv")
united_states_first_half_count = len(united_states) // 2
united_states_first_half_count

428

In [14]:
united_states_1 = united_states.iloc[:united_states_first_half_count]
united_states_1

,title,date,text
0,JPMorgan Chase & Co. - Primary Offering Prospe...,2023-08-21,Access the original document here\n\nPrimary O...
1,Bunge Ltd. - Quarterly Report for Quarter Endi...,2023-08-09,Access the original document here\n\nQuarterly...
2,Capital Group Core Plus Income ETF - Post-Effe...,2023-08-08,Access the original document here\n\nPost-Effe...
3,AB Active ETFs Inc. - Post-Effective Amendment...,2023-08-07,Access the original document here\n\nPost-Effe...
4,Kiniksa Pharmaceuticals Ltd. - Quarterly Repor...,2023-08-01,Access the original document here\n\nQuarterly...
...,...,...,...
423,"UPDATE 3-Standing against tide, Japan sells ye...",2002-06-26,"TOKYO, June 26 (Reuters) - Japanese authoriti..."
424,CHRONOLOGY-History of central bank interventi...,2002-06-04,"LONDON, June 4 (Reuters) - The following is a..."
425,BOJ goes solo with yen-selling intervention ...,2002-06-01,"NEW YORK, May 31 (Reuters) - Japan's monetary..."
426,ANALYSIS-BOJ goes solo with yen-selling inter...,2002-06-01,"NEW YORK, May 31 (Reuters) - Japan's monetary..."


In [19]:
# Due to Github limits on maximum file size (100mb) 2 of the output files (United_States and China) will be manually split in 2.
# Code will be updated later to automatically handle this

united_states = pd.read_csv("./output/United_States.csv")
united_states_first_half_count = len(united_states) // 2

united_states_1 = united_states.iloc[:150]
united_states_2 = united_states.iloc[150:]
united_states_1.to_csv("./output/united_states_1.csv", index=False)
united_states_2.to_csv("./output/united_states_2.csv", index=False)



china = pd.read_csv("./output/China.csv")
china_first_half_count = len(china) // 2

china_1 = china.iloc[:100]
china_2 = china.iloc[100:]
china_1.to_csv("./output/chinas_1.csv", index=False)
china_2.to_csv("./output/chinas_2.csv", index=False)